Q1: How many lesson pages

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [ ]:
len(documents)
# Answer to Q1: There are 72 lesson pages in the dataset

72

Q2: Indexing and Searching

In [8]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [6]:
#generating indexed FAQ documents
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

# Answer to Q2: The filename of the 1st result is "01-agentic-rag/lessons/14-agentic-loop.md"

Q3: RAG

In [33]:
# for Q3; modifying RAGBase class to use the new index and search method
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'content': 1.1}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append('Content: ' + doc['content'])
            lines.append('Filename: ' + doc['filename'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage

In [36]:
#How does the agentic loop keep calling the model until it stops?
#index = build_index(documents)
assistant = RAGBase(index, openai_client)

answer, usage = assistant.rag('How does the agentic loop keep calling the model until it stops?')
print("Answer: ", answer)

print("Input Tokens: ", usage.input_tokens)

#Answer to Q3: Approx 7000 input tokens are used to retrieve this answer

Answer:  The loop keeps calling the model by checking whether the response contains any `function_call` items. If it does, the code runs the tool, appends the tool result to the message history, and calls the model again.

It stops when the model returns a response with no function calls:

- `has_function_calls = True` if any tool call appears
- `while True` repeats the API call
- `break` happens when `has_function_calls == False`

So the agentic loop is basically: call model → run tools if needed → send results back → repeat until the model answers directly.
Input Tokens:  7135


Q4: Chunking

In [37]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
print("Number of chunks: ", len(chunks))

#Answer to Q4: The number of chunks generated is 295

Number of chunks:  295


Q5: RAG with Chunking

In [39]:
# index the chunks
index.fit(chunks)

In [ ]:
# point RAG at chunk index
chunky_assistant = RAGBase(index, openai_client)

answer, usage = chunky_assistant.rag('How does the agentic loop keep calling the model until it stops?')
print("Answer: ", answer)

print("Input Tokens: ", usage.input_tokens)

#Answer to  Q5: the number of input tokens decreased by approx 3X (from 7000 to 2300) after chunking the documents

Answer:  The loop keeps calling the model inside a `while True` loop and checks whether any `function_call` items came back in `response.output`.

- If there is at least one function call, it runs the tool, appends the result to `messages`, and loops again.
- If there are no function calls on that turn, it `break`s and stops.

So the stop condition is: **no function calls this turn means the model is done**.
Input Tokens:  2318


Q6: Turning the RAG with chunked index into an agent

In [ ]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer


In [ ]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

question = "How does the agentic loop work, and how is it different from plain RAG?"